<a href="https://colab.research.google.com/github/Blackthornedejavre/GoogleColap/blob/main/Catastro/Catastro_Procesamiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### MONTE A TRABAJAR

In [ ]:
# Variables
monte = "Presnes"

# ESTRUCTURA DIRECTORIOS MONTE NUEVO



Generamos la estructura de directorios dentro de Google Drive

In [ ]:
import os
from shutil import copyfile

# Crear directorios
directorio_monte = "/content/drive/MyDrive/Catastro/Catastro_Analisis/Actuacion_Catastro/" + monte
os.makedirs(directorio_monte, exist_ok=True)

dirConsulta_monte_0 = directorio_monte + "/0.Descarga datos del Intersect"
os.makedirs(dirConsulta_monte_0, exist_ok=True)

dirConsulta_monte_1 = directorio_monte + "/1.Archivos para consulta (Semilla)"
os.makedirs(dirConsulta_monte_1, exist_ok=True)

dirConsulta_monte_1_1 = dirConsulta_monte_1 + "/Semilla"
os.makedirs(dirConsulta_monte_1_1, exist_ok=True)

dirConsulta_monte_1_2 = dirConsulta_monte_1 + "/Envio a ServMontes"
os.makedirs(dirConsulta_monte_1_2, exist_ok=True)

dirConsulta_monte_2 = directorio_monte + "/2.Resultados Consulta Catastral Masiva"
os.makedirs(dirConsulta_monte_2, exist_ok=True)

dirConsulta_monte_3 = directorio_monte + "/3.Prefiltrado"
os.makedirs(dirConsulta_monte_3, exist_ok=True)

dirConsulta_monte_4 = directorio_monte + "/4.Filtrado datos"
os.makedirs(dirConsulta_monte_4, exist_ok=True)

dirConsulta_monte_5 = directorio_monte + "/5.Union_GIS"
os.makedirs(dirConsulta_monte_5, exist_ok=True)

dirConsulta_monte_6 = directorio_monte + "/6.Sintesis Catastral"
os.makedirs(dirConsulta_monte_6, exist_ok=True)

El siguiente paso no es necesario ya que directamente se genera el codigo en .xml y .zip

In [ ]:

# Mover archivos
filestocopy = "/content/drive/MyDrive/Catastro/General/Semilla_Raw/Semilla.xlsx"
dir_destino = dirConsulta_monte_1_1 + "/"
copyfile(filestocopy, dir_destino + "Semilla.xlsx")

filestocopy_2 = "/content/drive/MyDrive/Catastro/General/Semilla_Raw/Plantilla_ConsultaMasiva.txt"
dir_destino_2 = dirConsulta_monte_1_1 + "/"
copyfile(filestocopy_2, dir_destino_2 + "ConsultaMasiva.txt")

filestocopy_3 = "/content/drive/MyDrive/Catastro/General/Semilla_Raw/Plantilla_en_Blanco.xlsx"
dir_destino_3 = dirConsulta_monte_2 + "/"
copyfile(filestocopy_3, dir_destino_3 + "RC_" + monte + ".xlsx")

# Renombrar archivo
sinrenombrar = dirConsulta_monte_1_1 + "/Semilla.xlsx"
renombrado = dirConsulta_monte_1_1 + "/Semilla_" + monte + ".xlsx"
os.rename(sinrenombrar, renombrado)

sinrenombrar_2 = dirConsulta_monte_1_1 + "/ConsultaMasiva.txt"
renombrado_2 = dirConsulta_monte_1_1 + "/ConsultaMasiva_" + monte + ".txt"
os.rename(sinrenombrar_2, renombrado_2)

sinrenombrar_3 = dirConsulta_monte_2 + "/RC_" + monte + ".xlsx"
renombrado_3 = dirConsulta_monte_2 + "/Plantilla_en_Blanco_" + monte + ".xlsx"
os.rename(sinrenombrar_3, renombrado_3)

# Consulta Masiva

Para poder obtener un listado de referencias catatsrales hace falta guardar el shapefile de las parcelas en particular el archivo .dbf en la carpeta 0.Descarga datos del Intersect. El nombre del shape al guardar tiene que ser RF_{Monte}

In [ ]:
!pip install geopandas fiona shapely pyproj
import geopandas as gpd

In [ ]:

# Ruta al archivo DBF
ruta_archivo = f"/content/drive/MyDrive/Catastro/Catastro_Analisis/Actuacion_Catastro/{monte}/0.Descarga datos del Intersect/Previo_{monte}.dbf"

# Leer el archivo DBF con GeoPandas
dataframe = gpd.read_file(ruta_archivo)

# Extraer los valores de la columna "nationaCA"
valores_nationalCa = dataframe["nationalCa"].tolist()

# Imprimir los valores
for valor in valores_nationalCa:
    print(valor)

In [ ]:
import os
import xml.etree.ElementTree as ET
from xml.dom import minidom
from datetime import datetime

# Assuming you already have the valores_nationalCa list

# Create the root element without attributes
root = ET.Element("LISTADATOS")
# Set the xmlns and xmlns:xsi attributes on the root element
root.set("xmlns", "http://www.catastro.meh.es/")
root.set("xmlns:xsi", "http://www.w3.org/2001/XMLSchema-instance")

# Get the current date
current_date = datetime.now().strftime("%d/%m/%Y")

# Add the FEC element with the current date
fec_element = ET.SubElement(root, "FEC")
fec_element.text = current_date

# Add the FIN element
fin_element = ET.SubElement(root, "FIN")
fin_element.text = "Consulta por Referencia Catastral"

# Add the DAT elements for each value in valores_nationalCa
for valor in valores_nationalCa:
    dat_element = ET.SubElement(root, "DAT")
    rc_element = ET.SubElement(dat_element, "RC")
    rc_element.text = valor

# Create the XML tree
xml_tree = ET.ElementTree(root)

# Define the directory where you want to save the XML file
output_directory =f"/content/drive/MyDrive/Catastro/Catastro_Analisis/Actuacion_Catastro/{monte}/1.Archivos para consulta (Semilla)/Envio a ServMontes"
# Ensure the directory exists; create it if it doesn't
os.makedirs(output_directory, exist_ok=True)

# Specify the XML file name
xml_file_name = f"ConsultaMasiva_{monte}.xml"

# Construct the full file path
xml_file_path = os.path.join(output_directory, xml_file_name)

# Write the XML to the file
xml_tree.write(xml_file_path, encoding="utf-8", xml_declaration=True)

# Format the XML file to make it more readable (optional)
xml_dom = minidom.parse(xml_file_path)
formatted_xml = xml_dom.toprettyxml(indent="  ")

# Write the formatted XML back to the file
with open(xml_file_path, "w", encoding="utf-8") as f:
    f.write(formatted_xml)

# Print a message with the number of records written to the file
print(f"Numero de registros a consultar en el Servicio de Montes:{len(valores_nationalCa)}")


Numero de registros a consultar en el Servicio de Montes:2077


Una vez ya tenemos el archivo .xml generado lo comprimos en .zip en el mismo directorio de salida

In [ ]:
import zipfile
# Create a zip archive and add the formatted XML file to it
zip_file_name = f"ConsultaMasiva_{monte}.zip"
zip_file_path = os.path.join(output_directory, zip_file_name)

with zipfile.ZipFile(zip_file_path, "w") as zip_file:
    zip_file.write(xml_file_path, os.path.basename(xml_file_path))

ya podemos descargar todo lo que vamos a enviar al Servicio de Montes para realizar la consulta.


# Agrupar por APN


Una vez recibido del servicio de montes la consulta catastral se debe hacer una copia en el siguiente  f"/content/drive/MyDrive/Catastro/Catastro_Analisis/Actuacion_Catastro/{monte}/2.Resultados Consulta Catastral Masiva/RC_{monte}.xlsx"


In [ ]:
# Variables
monte = "Las Morteras"

In [ ]:
#Instalar Librerias
!pip install pandas
!pip install openpyxl

#Importar Librerias
import pandas as pd

# Rutas de archivos
directorio = f"/content/drive/MyDrive/Catastro/Catastro_Analisis/Actuacion_Catastro/{monte}/2.Resultados Consulta Catastral Masiva/RC_{monte}.xlsx"


# Leer el archivo Excel y obtener los nombres de las columnas
datos = pd.read_excel(directorio)
nombres_encabezados = datos.columns.tolist()

# Filtrar los nombres que comienzan por "NIF"
var_NIF = [encabezado for encabezado in nombres_encabezados if encabezado.startswith("NIF")]

# Solo tomar la segunda columna de la lista var_NIF
if len(var_NIF) > 1:
    var_NIF = [var_NIF[1]]
print(var_NIF)

['NIF4']


In [ ]:

# Filtrar los nombres que corresponden a las columnas a mostrar
columnas_mostrar = ['RC', 'SUF', 'APN', 'DFT1', 'DFT2'] + var_NIF

# Filtrar las filas donde la columna 'APN' no es nula
datos_filtrados = datos[columnas_mostrar].dropna(subset=['APN'])

# Agrupar por 'APN' y obtener una lista de 'RC' asociados a cada 'APN'
Info_Titulares = datos_filtrados.groupby('APN').agg({
    'RC': lambda x: ', '.join(sorted(set(x))),  # Combine unique 'RC' values into a comma-separated string
    'SUF': 'first',   # Take the first 'SUF' value for each 'APN'
    'DFT1': 'first',  # Take the first 'DFT1' value for each 'APN'
    'DFT2': 'first',  # Take the first 'DFT2' value for each 'APN'
    **{col: 'first' for col in var_NIF},  # Take the first value for each NIF column
}).reset_index()

# Mostrar el resultado
display(Info_Titulares)

,APN,RC,SUF,DFT1,DFT2,NIF4
0,VECINOS DE SANTA MARIA DE LLANUCES,"33053A01300348, 33053A01300396, 33053A01400056...",NaN,LG MURIELLOS ABAJO 103(A),33117 QUIROS (ASTURIAS),NaN


In [ ]:
# Group by y summarize en group_by_RF_2
group_by_RF_2 = datos[['RC', 'APN']].dropna(subset=['APN']).groupby('RC').agg({'APN': lambda x: ', '.join(pd.unique(x))}).reset_index()
UnaRCVariostitulares = group_by_RF_2.rename(columns={'APN': 'APNs'})
display(UnaRCVariostitulares)


,RC,APNs
0,33053A01300348,VECINOS DE SANTA MARIA DE LLANUCES
1,33053A01300396,VECINOS DE SANTA MARIA DE LLANUCES
2,33053A01400056,VECINOS DE SANTA MARIA DE LLANUCES
3,33053A01400075,VECINOS DE SANTA MARIA DE LLANUCES


In [ ]:
# Guardado en archivos Excel
Guardar_excel_Info_Titulares = f"/content/drive/MyDrive/Catastro/Catastro_Analisis/Actuacion_Catastro/{monte}/4.Filtrado datos/Filtrado_{monte}_Info_Titulares_colab.xlsx"
Info_Titulares.to_excel(Guardar_excel_Info_Titulares, index=True)

Guardar_excel_UnaRCVariostitulares = f"/content/drive/MyDrive/Catastro/Catastro_Analisis/Actuacion_Catastro/{monte}/4.Filtrado datos/Filtrado_{monte}_UnaRCVariostitulares_colap.xlsx"
UnaRCVariostitulares.to_excel(Guardar_excel_UnaRCVariostitulares, index= False)

# Guardado en archivos CSV
Guardar_csv_Info_Titulares = f"/content/drive/MyDrive/Catastro/Catastro_Analisis/Actuacion_Catastro/{monte}/4.Filtrado datos/Filtrado_{monte}_Info_Titulares_colab.csv"
Info_Titulares.to_csv(Guardar_csv_Info_Titulares, index=False)

Guardar_csv_UnaRCVariostitulares = f"/content/drive/MyDrive/Catastro/Catastro_Analisis/Actuacion_Catastro/{monte}/4.Filtrado datos/Filtrado_{monte}_UnaRCVariostitulares_colap.csv"
UnaRCVariostitulares.to_csv(Guardar_csv_UnaRCVariostitulares, index=False)

Ahora se puede categorizar el tipo de propiedad en "Privada", " Publica" "Desconocida"

In [ ]:
import pandas as pd
import os

# Directory for Excel and CSV files
excel_file_path  = f"/content/drive/MyDrive/Catastro/Catastro_Analisis/Actuacion_Catastro/{monte}/4.Filtrado datos/Filtrado_{monte}_UnaRCVariostitulares_colap.xlsx"

# Words dictionary
words_dict = {"VECINOS", "VECINAL", "CONFEDERACION", "CAMINOS", "AYUNTAMIENTO","JUNTA"}

# Read the Excel file
Excel_UnifRC_colap = pd.read_excel(excel_file_path)

# Function to map APNs to Tipo_Propiedad
def map_tipo_propiedad(apn_value):
    if apn_value == "EN INVESTIGACION":
        return "DESCONOCIDO"
    elif any(word.lower() in apn_value.lower() for word in words_dict):
        return "PÚBLICA"
    else:
        return "PRIVADA"

# Create the new "Tipo_Propiedad" column based on the mapping function
Excel_UnifRC_colap["Tipo_Propiedad"] = Excel_UnifRC_colap["APNs"].map(map_tipo_propiedad)


# Directory for Excel and CSV files
output_directory = f"/content/drive/MyDrive/Catastro/Catastro_Analisis/Actuacion_Catastro/{monte}/5.Union_GIS"
output_directory_Excel = os.path.join(output_directory, f"Filtrado_{monte}_UnaRCVariostitulares_TP_colab.xlsx")
output_directory_CSV = os.path.join(output_directory, f"Filtrado_{monte}_UnaRCVariostitulares_TP_colab.csv")

# Save the DataFrame to Excel and CSV files
Excel_UnifRC_colap.to_excel(output_directory_Excel, index= False)
Excel_UnifRC_colap.to_csv(output_directory_CSV, index=False,encoding='utf-8 sig')